In [8]:
from ASKpipeline import build_verification_graph
import pandas as pd

In [16]:
names = ['mintaka', 'qald', 'hotpot']

variants = ['small','base','large']
norm_files = [f'flan-t5-{variant}' for variant in variants]
vanilla_files = [f'vanilla-flan-t5-{variant}' for variant in variants]

files = norm_files + vanilla_files

file_name = files[0]


dataframes = {name: pd.read_csv(f'./LLM_answers/{file_name}/LLM_Answers_{name}.csv') for name in names}

if file_name.startswith('vanilla'):
    dataframes = {name: pd.read_csv(f'./LLM_answers/{file_name}/Vanilla_LLM_Answers_{name}.csv') for name in names}




In [17]:
file_name

'flan-t5-small'

In [21]:
from typing import Dict, List
import pandas as pd

def run_verification_on_datasets(
    datasets: Dict[str, pd.DataFrame],
    question_col: str = "question",
    answer_col: str = "answer",
    build_graph_fn=None,
    show_progress: bool = True,
) -> Dict[str, List[str]]:
    """
    Runs the verification graph on each row of each dataframe.
    Returns: {dataset_name: [verdicts...]} in row order.
    """
    if build_graph_fn is None:
        from ASKpipeline import build_verification_graph
        build_graph_fn = build_verification_graph

    graph = build_graph_fn()
    results: Dict[str, List[str]] = {}

    for name, df in datasets.items():
        verdicts: List[str] = []
        iterator = df.itertuples(index=False)
        if show_progress:
            try:
                from tqdm import tqdm
                iterator = tqdm(list(iterator), desc=f"Verifying {name}")
            except Exception:
                pass

        for row in iterator:
            row_dict = row._asdict()
            state = {
                "question": row_dict.get(question_col, ""),
                "answer": row_dict.get(answer_col, ""),
            }
            try:
                out = graph.invoke(state)
                verdicts.append(out.get("verdict", "Not Found"))
            except Exception:
                verdicts.append("ERROR")

        results[name] = verdicts

    return results
